In [105]:
# from sympy import Matrix, symbols, log, exp, simplify, Mul, Pow
import jax
import jax.numpy as jnp
import numpy as np
from tca_openflux import TCA_OpenFLux, objective_wssr, objective_clr_wssr
from emu_example import closure, clr_func, clr_inv_func

import optimistix

In [106]:
v_free = {"v8": 0.215, "v1": 1, "v19": 0.05, "v20": 0.03, "v16": 0.07, "v17": 0.23, "v18": 0.12, "v2": 0.201, "v9": 0.296}
fixed = jnp.array([v_free["v1"], v_free["v16"], v_free["v17"], v_free["v18"], v_free["v19"], v_free["v20"]])
# parameters = jnp.array([v8,v19,v20,v16,v17,v18,v2,v9])
parameters = jnp.array([v_free["v8"],v_free["v2"],v_free["v9"]])
data = TCA_OpenFLux(parameters=parameters, fixed=fixed)

# Simulations

In [ ]:
rng = np.random.default_rng(seed=1234)
key = jax.random.PRNGKey(1234)
sims = 1000

total_mol = 1e5
total_isos = jax.tree.map(lambda x: x * total_mol, data)
cv = 0.2
norm_sigma_cv = jax.tree.map(lambda x: x * cv, total_isos)

key, key_sim = jax.random.split(key)
norm_sim = jax.tree.map(
    lambda mean, sigma: mean + sigma * jax.random.normal(key_sim, (sims, mean.shape[0])),
    total_isos,
    norm_sigma_cv,
)

In [128]:
# Simulated reads closed to 1
iso_proportions = jax.tree.map(
    lambda mat: closure(mat),
    norm_sim,
)

# Check that all sum to 1
jax.tree.map(
    lambda x: jnp.all(jnp.isclose(jnp.sum(x, axis=-1), 1.0, rtol=1e-10, atol=1e-10)),
    iso_proportions
)

{'aspx_1234': Array(True, dtype=bool),
 'lysx_123456': Array(True, dtype=bool),
 'suc_1234': Array(True, dtype=bool),
 'valx_12345': Array(True, dtype=bool)}

In [129]:
mean_sim = jax.tree.map(
    lambda x: jnp.mean(x, axis=0),  # mean over isotopolouges
    iso_proportions
)

std_sim = jax.tree.map(
    lambda x: jnp.std(x, axis=0),  # standard deviation over isotopolouges
    iso_proportions
)

D_sim = jax.tree.map(
    lambda x: jnp.diag(x), 
    std_sim
)

In [130]:
solver = optimistix.BFGS(
    rtol=1e-12, atol=1e-12, verbose=frozenset({"step"})
)

In [131]:
# initial_guess = jnp.array([0.043 , 0.01  , 0.006 , 0.014 , 0.046 , 0.024 , 0.0402, 0.0592])
initial_guess = jnp.array([1, 1, 1]) # v8, v2, v9

In [132]:
result_optimistix = optimistix.minimise(objective_wssr, y0=initial_guess, args=(mean_sim, D_sim, fixed), solver=solver, max_steps=10000)

In [133]:
result_optimistix.value

Array([0.22399349, 0.20240499, 0.30363819], dtype=float64)

# CLR transformed

In [134]:
iso_clr = jax.tree.map(
    lambda x: clr_func(x),
    iso_proportions
)

# Check that all sum to 0
jax.tree.map(
    lambda x: jnp.all(jnp.isclose(jnp.sum(x, axis=-1), 0.0, rtol=1e-10, atol=1e-10)),
    iso_clr
)

{'aspx_1234': Array(True, dtype=bool),
 'lysx_123456': Array(True, dtype=bool),
 'suc_1234': Array(True, dtype=bool),
 'valx_12345': Array(True, dtype=bool)}

In [135]:
mean_clr = jax.tree.map(
    lambda x: jnp.mean(x, axis=0),  # mean over bins
    iso_clr
)

std_clr = jax.tree.map(
    lambda x: jnp.std(x, axis=0),  # mean over isotopolouges
    iso_clr
)

D_clr = jax.tree.map(
    lambda x: jnp.diag(x), 
    std_clr
)

In [136]:
mean_clr_inv = jax.tree.map(
    lambda x: clr_inv_func(x),
    mean_clr
)

In [137]:
result_optimistix_clr = optimistix.minimise(objective_clr_wssr, y0=initial_guess, args=(mean_clr, D_clr, fixed), solver=solver, max_steps=10000)

In [138]:
print("True parameters: ", parameters)
print("Euclidean estimated parameters: ", result_optimistix.value)
print("CLR transformed estimated parameters: ", result_optimistix_clr.value)

True parameters:  [0.215 0.201 0.296]
Euclidean estimated parameters:  [0.22399349 0.20240499 0.30363819]
CLR transformed estimated parameters:  [0.21283379 0.20105222 0.29512406]
